[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W01_From_a_biological_question_to_defensible_evidence.ipynb)

# Week 01 | From a biological question to defensible evidence

**Core practical: 45 minutes.** Calculate a sequence summary two ways, decide which number you would report, and keep the provenance of both.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
Recall: genes can produce RNA or protein products; RNA counts are not direct protein measurements.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. The cell refuses to run while the placeholder is still there, because the placeholder deals every student the same dataset. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
if COURSE_ID == "demo-001":
    raise ValueError(
        "COURSE_ID is still the placeholder. Replace it with your assigned pseudonym and "
        "run this cell again; the placeholder deals the same dataset to the whole class."
    )
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Prediction
You will be dealt two synthetic regions of equal length: one drawn at GC-rich composition, one at AT-rich composition. Both contain some positions an assembly could not resolve, written `N`.

Before you run anything, commit to two predictions: which region will have the higher GC fraction, and whether counting the `N` positions in the denominator would push a GC fraction up or down. Then copy both into `PREDICTION` in the response cell.

## Guided investigation (20 min)
1. Predict GC for ACGTGC before execution.
2. Run the calculation and then the edge-case tests.
3. Read your two assigned regions. Check your prediction about which one is GC-rich.
4. Two supplied functions describe themselves the same way and return different numbers on your data. Decide which one you would report, and how you decided.
5. Save a short interpretation and your disclosure.

In [ ]:
def gc_known(sequence):
    """GC fraction over KNOWN bases: N positions are excluded from the denominator."""
    seq = sequence.upper()
    if any(b not in "ACGTN" for b in seq):
        raise ValueError("Only A, C, G, T and N are accepted in this toy exercise")
    known = [b for b in seq if b in "ACGT"]
    return sum(b in "GC" for b in known) / len(known) if known else None

def gc_all(sequence):
    """Supplied alternative. Its author also called it 'the fraction of G or C bases'."""
    seq = sequence.upper()
    return sum(b in "GC" for b in seq) / len(seq)

assert abs(gc_known("ACGTGC") - 4/6) < 1e-12
assert gc_known("GCNN") == 1.0
assert gc_known("NN") is None

def synthetic_region(rng, length, gc_target, n_unknown):
    """A SYNTHETIC sequence drawn at a stated G/C probability, with unresolved N positions.

    This is not an excerpt from any genome. It imitates one property of real DNA only:
    that different kinds of region differ systematically in base composition.
    """
    at, gc = (1 - gc_target) / 2, gc_target / 2
    seq = rng.choices("ACGT", weights=[at, gc, gc, at], k=length)
    for position in rng.sample(range(length), n_unknown):
        seq[position] = "N"
    return "".join(seq)

# In a real genome a promoter / CpG island is GC-rich and a deep intron is not.
# Region A and region B are synthetic stand-ins for that contrast, drawn from your seed.
N_UNKNOWN = rng.randint(6, 18)
REGION_A = synthetic_region(rng, 120, gc_target=0.68, n_unknown=N_UNKNOWN)  # island-like
REGION_B = synthetic_region(rng, 120, gc_target=0.36, n_unknown=N_UNKNOWN)  # intron-like

RESULTS = {"data_status": "SYNTHETIC", "region_length": 120, "n_unknown": N_UNKNOWN}
for label, seq in (("A", REGION_A), ("B", REGION_B)):
    RESULTS[f"region_{label}"] = {
        "sequence": seq,
        "GC_known": round(gc_known(seq), 4),
        "GC_all": round(gc_all(seq), 4),
    }

print("Worked example, GC of ACGTGC:", gc_known("ACGTGC"))
for label in ("A", "B"):
    record = RESULTS[f"region_{label}"]
    print(f"\nRegion {label}: {N_UNKNOWN} unresolved of 120 positions")
    print(" ", record["sequence"])
    print(f"  gc_known = {record['GC_known']}    gc_all = {record['GC_all']}")

optional_plot(["A gc_known", "A gc_all", "B gc_known", "B gc_all"],
              [RESULTS["region_A"]["GC_known"], RESULTS["region_A"]["GC_all"],
               RESULTS["region_B"]["GC_known"], RESULTS["region_B"]["GC_all"]],
              "GC fraction", "Two regions, two definitions of the denominator")

## Explain the evidence (10 min)
**Q1.** State the numerator and denominator for ACGTGC and GCNN.

**Q2.** Report `gc_known` for both of your regions. Which region is GC-rich, did that match your prediction, and what kind of genomic region would each one be a stand-in for?

**Q3.** `gc_known` and `gc_all` describe themselves the same way and return different numbers on your data. Which would you report? Say exactly what the other one counts, in which direction it is wrong, and which one of the three `assert` lines above would have caught it.

**Q4.** Can a GC fraction identify a species or establish that a sequence is biologically functional? Explain.

In [ ]:
PREDICTION = ""  # Fill before the analysis: which region is GC-rich, and which way N shifts a fraction.
RESPONSES = {"Q1": "", "Q2": "", "Q3": "", "Q4": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, restart the runtime and run all. Download the `.ipynb` and generated summary JSON. In Colab the JSON is in the Files sidebar. Upload both to the course LMS assignment. Do not email patient data. A completion flag checks presence of responses, not scientific correctness. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":1, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W01_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

## Paper / device-free route
Use ACGTGC, GCNN and NN. Count with a pencil. Write 4/6, 2/2 and undefined.

Then take this twenty-base region, which stands in for your dealt data:

```text
ATGCCGGCATNNGCATGCAT
```

Count G and C once, then write the two fractions the two functions would return: G+C over known bases, and G+C over all twenty positions. The numerator is the same in both. Complete the same four questions.

## Optional extension
Optional: turn one `N` in region B into a `G`. Before running it, predict how far each of the two functions moves, and which moves further.

## Sources
- [S01] Alberts et al. Molecular Biology of the Cell, 4th ed. Foundational cell biology chapters. https://www.ncbi.nlm.nih.gov/books/NBK21054/
- [S02] GENCODE Human Release 50 and release-specific statistics. https://www.gencodegenes.org/human/stats_50.html
- [S05] Python 3 tutorial: introduction and control flow. https://docs.python.org/3/tutorial/
- [S10] DESeq2: Analyzing RNA-seq data with DESeq2, release vignette. https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html
- [S20] Turkish Personal Data Protection Law No. 6698, official English translation with 2024 amendments. https://www.kvkk.gov.tr/Icerik/6649/Personal-Data-Protection-Law
- [S23] ICMJE recommendations: use of AI by authors. https://www.icmje.org/recommendations/browse/artificial-intelligence/ai-use-by-authors.html